In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../Dataset/raw_data/SuperstoreSales.csv", encoding="latin1")

print("Original shape:", df.shape)
df.head()

Original shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [2]:
#Convert Order Date and Ship Date to real datetime
# Before conversion - check current dtype
print("Before conversion:")
print(df[['Order Date', 'Ship Date']].dtypes)

# Convert to datetime
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%m/%d/%Y')

# After conversion - confirm
print("\nAfter conversion:")
print(df[['Order Date', 'Ship Date']].dtypes)

Before conversion:
Order Date    object
Ship Date     object
dtype: object

After conversion:
Order Date    datetime64[ns]
Ship Date     datetime64[ns]
dtype: object


In [3]:
#Verify Product Name whitespace (should already be clean, per our SQL finding — but let's independently confirm in Python)
whitespace_check = df['Product Name'].apply(lambda x: x != x.strip()).sum()
print("Product Name rows with whitespace:", whitespace_check)

# Apply trim as a safety measure regardless
df['Product Name'] = df['Product Name'].str.strip()

# Confirm after
whitespace_check_after = df['Product Name'].apply(lambda x: x != x.strip()).sum()
print("After trim:", whitespace_check_after)

Product Name rows with whitespace: 16
After trim: 0


In [4]:
#Handle Product_ID/Name situation in Python (documentation only, same decision as SQL)
# Check for Product_IDs mapped to multiple Product_Names (same check as SQL)
product_check = df.groupby('Product ID')['Product Name'].nunique()
mismatched_products = product_check[product_check > 1]

print("Number of Product_IDs with multiple names:", len(mismatched_products))
print("\nProduct_IDs affected:")
print(mismatched_products.index.tolist())

Number of Product_IDs with multiple names: 32

Product_IDs affected:
['FUR-BO-10002213', 'FUR-CH-10001146', 'FUR-FU-10001473', 'FUR-FU-10004017', 'FUR-FU-10004091', 'FUR-FU-10004270', 'FUR-FU-10004848', 'FUR-FU-10004864', 'OFF-AP-10000576', 'OFF-AR-10001149', 'OFF-BI-10002026', 'OFF-BI-10004632', 'OFF-BI-10004654', 'OFF-PA-10000357', 'OFF-PA-10000477', 'OFF-PA-10000659', 'OFF-PA-10001166', 'OFF-PA-10001970', 'OFF-PA-10002195', 'OFF-PA-10002377', 'OFF-PA-10003022', 'OFF-ST-10001228', 'OFF-ST-10004950', 'TEC-AC-10002049', 'TEC-AC-10002550', 'TEC-AC-10003832', 'TEC-MA-10001148', 'TEC-PH-10001530', 'TEC-PH-10001795', 'TEC-PH-10002200', 'TEC-PH-10002310', 'TEC-PH-10004531']


In [6]:
#let's finalize and save the cleaned dataset
# Final verification before saving
print("Final shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum().sum(), "total missing values")
print("\nDuplicate rows:", df.duplicated().sum())
print("\nDate dtypes confirmed:")
print(df[['Order Date', 'Ship Date']].dtypes)

# Save the cleaned dataset
df.to_csv("../Dataset/cleaned/superstore_cleaned.csv", index=False)
print("\n✅ Cleaned dataset saved successfully!")

Final shape: (9994, 21)

Missing values:
0 total missing values

Duplicate rows: 0

Date dtypes confirmed:
Order Date    datetime64[ns]
Ship Date     datetime64[ns]
dtype: object

✅ Cleaned dataset saved successfully!
